# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YassinKhodir/yassin-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds a simple rule for the **Refresh / Content Opportunity Scoring** lane. It uses only March 1–15 signals, then checks the ranked queue against the observed March 16–31 decline label. The rule supports human review; it does not prove that a refresh will cause recovery.

## 1. Check the two signals before writing the rule

The rule leans on two signals used in real content-opportunity logic:

1. **Search volume** — enough impressions make a review potentially valuable. This is the signal behind a quick-win volume floor.
2. **Average position** — pages outside the strongest positions may have more room for improvement. This is related to position-aware CTR review.

I bucket each signal and print `n` so a small group cannot look more convincing than it is. The outcome is later impression decline, measured after the decision window.

In [1]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

required = {"duckdb": "duckdb", "pandas": "pandas", "numpy": "numpy"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import duckdb
import numpy as np
import pandas as pd

data_root = os.environ.get("FLYRANK_DATA_ROOT")
if data_root:
    REL = data_root.rstrip("/")
else:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = None
    if not hf_token:
        raise RuntimeError("Add a Hugging Face READ token as a Colab Secret named HF_TOKEN; never paste it into this public notebook.")
    REL = "hf://datasets/FlyRank/internship-warehouse"

con = duckdb.connect()
if not data_root:
    escaped_token = hf_token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
    del hf_token, escaped_token

MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

feature_query = f"""
WITH early AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS early_impressions,
        SUM(gsc_clicks) AS early_clicks,
        COUNT(*) AS early_observed_days,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS early_active_days,
        SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                 THEN gsc_avg_position * gsc_impressions ELSE 0 END)
          / NULLIF(SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                            THEN gsc_impressions ELSE 0 END), 0) AS early_avg_position,
        STDDEV_SAMP(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS early_position_volatility
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
late AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS late_impressions,
        COUNT(*) AS late_observed_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1, 2
)
SELECT
    e.*,
    1.0 * e.early_clicks / NULLIF(e.early_impressions, 0) AS early_ctr,
    CAST((1.0 * l.late_impressions / NULLIF(l.late_observed_days, 0))
         < 0.8 * (1.0 * e.early_impressions / NULLIF(e.early_observed_days, 0)) AS INTEGER)
      AS is_future_decline
FROM early e
JOIN late l USING (client_hash_id, content_hash_id)
"""

frame = con.sql(feature_query).df()
assert not frame.duplicated(["client_hash_id", "content_hash_id"]).any()
print(f"Eligible page rows: {len(frame):,}")
print(f"Represented clients: {frame['client_hash_id'].nunique()}")
print(f"Observed future-decline base rate: {frame['is_future_decline'].mean():.3f}")

Eligible page rows: 77,400
Represented clients: 38
Observed future-decline base rate: 0.345


In [2]:
frame["volume_bucket"] = pd.cut(
    frame["early_impressions"],
    [-1, 299, 999, 2999, 9999, np.inf],
    labels=["100–299", "300–999", "1k–3k", "3k–10k", "10k+"],
)
volume_check = (
    frame.groupby("volume_bucket", observed=False)
    .agg(n=("is_future_decline", "size"), decline_rate=("is_future_decline", "mean"))
    .reset_index()
)
volume_check["decline_rate"] = volume_check["decline_rate"].round(3)
print("Signal 1 — early impressions")
print(volume_check.to_string(index=False))
print("Verdict: MIXED — most buckets are similar, but the 10k+ group has a higher observed decline rate.")

frame["position_bucket"] = pd.cut(
    frame["early_avg_position"],
    [0, 3, 10, 20, 50, np.inf],
    labels=["top 3", "page 1", "positions 11–20", "positions 21–50", "50+"],
)
position_check = (
    frame.groupby("position_bucket", observed=False)
    .agg(n=("is_future_decline", "size"), decline_rate=("is_future_decline", "mean"))
    .reset_index()
)
position_check["decline_rate"] = position_check["decline_rate"].round(3)
print("\nSignal 2 — early average position")
print(position_check.to_string(index=False))
print("Verdict: MIXED — positions 21–50 have the highest observed rate, but the pattern is not monotonic.")

Signal 1 — early impressions
volume_bucket     n  decline_rate
      100–299 24475         0.345
      300–999 25941         0.344
        1k–3k 16898         0.339
       3k–10k  8210         0.352
         10k+  1876         0.399
Verdict: MIXED — most buckets are similar, but the 10k+ group has a higher observed decline rate.

Signal 2 — early average position
position_bucket     n  decline_rate
          top 3 10137         0.323
         page 1 38761         0.339
positions 11–20 13309         0.319
positions 21–50 13494         0.407
            50+  1699         0.340
Verdict: MIXED — positions 21–50 have the highest observed rate, but the pattern is not monotonic.


## 2. Build the ranked queue

**Rule in plain words:** review pages first when they have substantial early-window impressions and rank in positions 4–50, especially positions 21–50. High volume raises possible value; a weaker-but-visible position raises possible room to improve.

The score uses fixed points, not fitted weights:

- Volume points: 0 for 100–299 impressions, then 1/2/3/4 as volume rises.
- Position points: 0 for top 3 or deeper than 50, 1 for positions 4–10, 2 for 11–20, and 3 for 21–50.
- One reason code records the main reason for each action.

This score is frozen before the Week-5 model. It never uses March 16–31 values, the label, a product flag, or either ID as a feature.

In [3]:
frame["volume_points"] = pd.cut(
    frame["early_impressions"],
    [-1, 299, 999, 2999, 9999, np.inf],
    labels=[0, 1, 2, 3, 4],
).astype(int)

frame["position_points"] = np.select(
    [
        frame["early_avg_position"].between(20, 50, inclusive="right"),
        frame["early_avg_position"].between(10, 20, inclusive="right"),
        frame["early_avg_position"].between(3, 10, inclusive="right"),
    ],
    [3, 2, 1],
    default=0,
)
frame["baseline_score"] = frame["volume_points"] + frame["position_points"]

frame["reason_code"] = np.select(
    [
        (frame["volume_points"] >= 3) & (frame["position_points"] == 3),
        frame["position_points"] == 3,
        frame["volume_points"] >= 3,
    ],
    ["high_volume_position_21_50", "visible_position_21_50", "high_volume_visible"],
    default="moderate_opportunity",
)
frame["action"] = np.where(frame["baseline_score"] >= 5, "review for refresh", "monitor")

queue = frame.sort_values(
    ["baseline_score", "early_impressions", "early_avg_position"],
    ascending=[False, False, True],
).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

output_dir = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / "baseline_action_score.csv"
queue.to_csv(queue_path, index=False)

def precision_at_k(data, k):
    return data.head(k)["is_future_decline"].mean()

base_rate = frame["is_future_decline"].mean()
metrics = {
    "eligible_rows": int(len(frame)),
    "represented_clients": int(frame["client_hash_id"].nunique()),
    "base_rate": round(float(base_rate), 6),
    "precision_at_10": round(float(precision_at_k(queue, 10)), 6),
    "precision_at_50": round(float(precision_at_k(queue, 50)), 6),
    "precision_at_100": round(float(precision_at_k(queue, 100)), 6),
}
with open(output_dir / "ml07_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Wrote {len(queue):,} ranked rows to {queue_path}")
print(pd.Series(metrics).to_string())

Wrote 77,400 ranked rows to work/outputs/baseline_action_score.csv
eligible_rows          77400.000000
represented_clients       38.000000
base_rate                  0.345465
precision_at_10            0.800000
precision_at_50            0.660000
precision_at_100           0.680000


## 3. Top-10 review

The table below hides the pseudonymous IDs because the review only needs rank-level evidence. Each row states the proposed action, why it ranked, my confidence, and a concrete reason the choice could be wrong.

In [4]:
top10 = queue.head(10).copy()
top10["confidence_note"] = np.where(
    top10["baseline_score"] >= 7,
    "high rule confidence; outcome still uncertain",
    "moderate rule confidence",
)
wrong_notes = [
    "the traffic may be seasonal rather than a lasting opportunity",
    "the page may already satisfy its search intent and need no refresh",
    "the query mix may explain the average position better than content quality",
    "high impressions may not match the client's business value",
    "a technical indexing issue may be the real problem",
    "the early window may include a short-lived demand spike",
    "competing pages may have changed during the outcome window",
    "the page may target a broad query where position is naturally unstable",
    "the content may already be scheduled for consolidation or removal",
    "March behavior may not represent the page's usual pattern",
]
top10["what_would_make_it_wrong"] = wrong_notes

review = top10[[
    "rank", "action", "reason_code", "baseline_score",
    "confidence_note", "what_would_make_it_wrong",
]].copy()
pd.set_option("display.max_colwidth", 90)
review

   rank  ...                                                    what_would_make_it_wrong
0     1  ...               the traffic may be seasonal rather than a lasting opportunity
1     2  ...          the page may already satisfy its search intent and need no refresh
2     3  ...  the query mix may explain the average position better than content quality
3     4  ...                  high impressions may not match the client's business value
4     5  ...                          a technical indexing issue may be the real problem
5     6  ...                     the early window may include a short-lived demand spike
6     7  ...                  competing pages may have changed during the outcome window
7     8  ...      the page may target a broad query where position is naturally unstable
8     9  ...           the content may already be scheduled for consolidation or removal
9    10  ...                   March behavior may not represent the page's usual pattern

[10 rows x 6 columns

## 4. Weak picks and leakage check

The weakest part of this baseline is that a high-volume page can rank near the top even when its business importance, query intent, content quality, and seasonality are unknown. Position 21–50 also had a higher observed decline rate, but the bucket pattern was mixed rather than steadily increasing. Those are reasons for human review, not proof that a refresh is needed.

The feature window ends on March 15. The March 16–31 decline label is used only after ranking to calculate precision, never inside the score. Client and content hashes are context fields only. No product flags, URLs, queries, later-window values, or label copies enter the rule.

In [5]:
score_inputs = {"volume_points", "position_points"}
forbidden_inputs = {
    "is_future_decline", "late_impressions", "client_hash_id", "content_hash_id",
    "trend_pct", "trend_direction",
}
assert score_inputs.isdisjoint(forbidden_inputs)
assert queue["baseline_score"].equals(queue["volume_points"] + queue["position_points"])

weak_pick_count = int((top10["early_position_volatility"].fillna(0) >= 10).sum())
print("Score inputs:", sorted(score_inputs))
print("Forbidden inputs used in score: none")
print("Top-10 picks with position volatility >= 10:", weak_pick_count)
print("Baseline frozen for the next assignment.")

Score inputs: ['position_points', 'volume_points']
Forbidden inputs used in score: none
Top-10 picks with position volatility >= 10: 0
Baseline frozen for the next assignment.


## Self-check

- [x] Two signal checks show bucket tables, `n`, and one-word verdicts
- [x] At least one checked signal is tied to real flag logic
- [x] One readable rule produces a score, one reason code, and an action
- [x] The notebook writes `work/outputs/baseline_action_score.csv`
- [x] All top 10 rows include what would make the choice wrong
- [x] Precision@K and the base rate use the observed later window
- [x] No future-window or label-derived value enters the score
- [x] The notebook runs top to bottom without errors
- [x] No client names, domains, URLs, private queries, or credentials appear